# ✦ LILY WAN 2.2 — Dual-T4 Studio — FIXED v3

Set **Internet ON** and select **GPU T4 x2**. This version hard-checks that Kaggle actually gave the session two T4s, isolates ComfyUI in its own virtual environment so pip cannot break/restart the notebook kernel, and skips RIFE during startup for reliability.


In [ ]:
import json, urllib.request, importlib.util, sys, subprocess, os
from pathlib import Path

print('✦ Lily Wan 2.2 Studio — safe startup v3')

# ------------------------------------------------------------
# 0) HARDWARE GUARD: this build is specifically for T4 x2.
# ------------------------------------------------------------
try:
    raw = subprocess.check_output([
        'nvidia-smi', '--query-gpu=name', '--format=csv,noheader'
    ], text=True, stderr=subprocess.STDOUT)
    gpu_names = [x.strip() for x in raw.splitlines() if x.strip()]
except Exception as e:
    raise RuntimeError(f'Could not detect Kaggle GPUs: {e}')

print('Detected GPUs:', gpu_names)
if len(gpu_names) != 2 or not all('T4' in name.upper() for name in gpu_names):
    raise RuntimeError(
        'WRONG KAGGLE ACCELERATOR. This notebook requires GPU T4 x2. '
        f'Kaggle actually gave this session: {gpu_names}. '
        'Stop this session, choose Accelerator → GPU T4 x2, then start a fresh session and run again.'
    )
print('✓ Correct hardware: dual T4 detected')

# ------------------------------------------------------------
# 1) Create an isolated environment for ComfyUI.
#    --system-site-packages reuses Kaggle CUDA/PyTorch instead
#    of downloading another giant torch build.
# ------------------------------------------------------------
VENV = Path('/kaggle/working/lily_comfy_env')
VENV_PY = VENV / 'bin' / 'python'
if not VENV_PY.exists():
    print('Creating isolated ComfyUI environment...')
    subprocess.run([sys.executable, '-m', 'venv', '--system-site-packages', str(VENV)], check=True)
print('✓ Isolated ComfyUI environment ready')

# ------------------------------------------------------------
# 2) Load the full studio snapshot, then patch it before exec.
# ------------------------------------------------------------
SOURCE_URL = 'https://raw.githubusercontent.com/benruiz1024-ops/hi/789b857e85738efdaec591f04de11bf76befe20d/LILY_WAN22_DUAL_T4_STUDIO.ipynb'
with urllib.request.urlopen(SOURCE_URL, timeout=60) as r:
    original = json.loads(r.read().decode('utf-8'))

code_cells = [c for c in original['cells'] if c.get('cell_type') == 'code']
if not code_cells:
    raise RuntimeError('Could not locate the studio code cell.')
code = ''.join(code_cells[0]['source'])

# Never blanket-upgrade Kaggle's own requests/pandas/jupyter stack.
bad_core = '''# Core helpers/UI.
pip_install("gradio>=5.20,<6", "huggingface_hub>=0.29", "requests>=2.32", "gdown>=5.2", "scikit-video")'''
safe_core = '''# Core helpers/UI — SAFE KAGGLE INSTALL.
import importlib.util
_missing = []
for _mod, _pkg in [("gradio", "gradio"), ("huggingface_hub", "huggingface_hub"), ("gdown", "gdown")]:
    if importlib.util.find_spec(_mod) is None:
        _missing.append(_pkg)
if _missing:
    run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "--no-input", *_missing], check=False)
else:
    print("✓ Kaggle core packages already present — no upgrades needed")'''
if bad_core not in code:
    raise RuntimeError('Safety patch target 1 was not found.')
code = code.replace(bad_core, safe_core, 1)

# Disable the old blanket-upgrade helper.
old_helper = '''def pip_install(*pkgs):
    run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", *pkgs])'''
new_helper = '''def pip_install(*pkgs):
    run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "--no-input", *pkgs], check=False)'''
if old_helper in code:
    code = code.replace(old_helper, new_helper, 1)

# Install ComfyUI requirements INSIDE THE VENV, never into Kaggle's notebook kernel.
old_req = 'run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], cwd=COMFY)'
new_req = '''print("Installing ComfyUI dependencies in isolated env (this can take a few minutes)...")
run([str(VENV_PY), "-m", "pip", "install", "--disable-pip-version-check", "--no-input", "--prefer-binary", "-r", "requirements.txt"], cwd=COMFY, check=True)
print("✓ ComfyUI dependencies installed without touching the Kaggle kernel")'''
if old_req not in code:
    raise RuntimeError('Could not locate the ComfyUI requirements installer to isolate it.')
code = code.replace(old_req, new_req, 1)

# MagCache requirements are optional, but if present install them into the same venv.
old_mag_req = 'run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)], check=False)'
new_mag_req = 'run([str(VENV_PY), "-m", "pip", "install", "--disable-pip-version-check", "--no-input", "--prefer-binary", "-r", str(req)], check=False)'
if old_mag_req in code:
    code = code.replace(old_mag_req, new_mag_req, 1)

# Launch ComfyUI with the isolated Python interpreter.
old_launch = '        sys.executable, "main.py",'
new_launch = '        str(VENV_PY), "main.py",'
if old_launch not in code:
    raise RuntimeError('Could not locate the ComfyUI launch interpreter.')
code = code.replace(old_launch, new_launch, 1)

# Skip Practical-RIFE at startup. It was the earlier Git clone hang.
rife_start = '# -------------------------\n# 4) Optional RIFE on GPU 1'
rife_end = '# -------------------------\n# 5) Start ComfyUI on GPU 0'
a = code.find(rife_start)
b = code.find(rife_end)
if a == -1 or b == -1 or b <= a:
    raise RuntimeError('Could not locate the RIFE block to disable it safely.')
no_rife = '''# -------------------------
# 4) Reliable startup mode: RIFE deferred
# -------------------------
RIFE_READY = False
print("✓ RIFE deferred — ffmpeg interpolation will be used for now")

'''
code = code[:a] + no_rife + code[b:]

print('✓ Dual-T4 hardware check passed')
print('✓ Kernel-safe isolated dependency patch applied')
print('✓ RIFE startup clone disabled')
print('✓ Starting Wan 2.2 setup...\n')

exec(compile(code, 'LILY_WAN22_DUAL_T4_STUDIO_SAFE_V3', 'exec'), globals(), globals())
